# 01 ROI FCN Training Control Panel v0.2

Thin operator-facing notebook for launching ROI-FCN training in a detached `tmux` session and evaluating saved runs.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
TRAINING_ROOT = NOTEBOOK_DIR.parent
SRC_ROOT = TRAINING_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

PYTHON_EXECUTABLE = sys.executable

print(f'training_root={TRAINING_ROOT}')
print(f'python={PYTHON_EXECUTABLE}')


training_root=/home/mitch/development/raccoon-ball/04_ROI-FCN/02_training
python=/home/mitch/development/raccoon-ball/.venv/bin/python


In [2]:
import html
from pathlib import Path
import threading
import traceback

import ipywidgets as widgets
from IPython.display import display

from roi_fcn_training_v0_1.config import EvalConfig, TrainConfig
from roi_fcn_training_v0_1.discovery import discover_dataset_references
from roi_fcn_training_v0_1.evaluate import evaluate_saved_run
from roi_fcn_training_v0_1.paths import find_training_root, preview_next_run_id, suggest_model_run_id
from roi_fcn_training_v0_1.tmux_launcher_v0_2 import (
    TMUX_CONTROL_PANEL_BUILD_V02,
    default_session_name,
    end_session,
    launch_session,
    list_sessions,
    plan_tmux_training_launch,
    read_log_tail,
    session_exists,
)
from roi_fcn_training_v0_1.topologies import get_topology_definition, list_topology_ids, list_topology_variants

training_root = find_training_root(TRAINING_ROOT)

preview_html = widgets.HTML()
status_html = widgets.HTML()
topology_help = widgets.HTML()

train_dataset_dropdown = widgets.Dropdown(description='Train Data')
validation_dataset_dropdown = widgets.Dropdown(description='Val Data')
refresh_datasets_button = widgets.Button(description='Refresh Datasets')

topology_ids = list_topology_ids()
topology_id_dropdown = widgets.Dropdown(
    description='Topology',
    options=[(value, value) for value in topology_ids],
    value=topology_ids[0],
)
default_variants = list_topology_variants(topology_ids[0])
topology_variant_dropdown = widgets.Dropdown(
    description='Variant',
    options=[(value, value) for value in default_variants],
    value=default_variants[0],
)

model_name_text = widgets.Text(description='Model Name', value='roi-fcn-tiny')
model_directory_text = widgets.Text(description='Model Dir', value='')
run_id_text = widgets.Text(description='Run ID', value='')
session_name_text = widgets.Text(description='Session', value='')
device_text = widgets.Text(description='Device', value='', placeholder='blank -> auto CUDA; CPU fallback disabled for training')

batch_size = widgets.BoundedIntText(description='Batch Size', value=16, min=1, max=4096)
epochs = widgets.BoundedIntText(description='Epochs', value=8, min=1, max=10000)
learning_rate = widgets.FloatText(description='LR', value=1e-3)
weight_decay = widgets.FloatText(description='Weight Decay', value=1e-5)
gaussian_sigma = widgets.FloatText(description='Sigma Px', value=2.5)
early_stopping_patience = widgets.BoundedIntText(description='Patience', value=4, min=1, max=10000)
progress_log_interval_steps = widgets.BoundedIntText(description='Log Every', value=50, min=1, max=1000000)
roi_width = widgets.BoundedIntText(description='ROI Width', value=300, min=1, max=4096)
roi_height = widgets.BoundedIntText(description='ROI Height', value=300, min=1, max=4096)
evaluation_max_visual_examples = widgets.BoundedIntText(description='Eval Visuals', value=12, min=0, max=512)

run_dir_text = widgets.Text(description='Run Dir', value='', placeholder='Predicted run directory or existing run to evaluate')
log_path_text = widgets.Text(description='Log Path', value='', placeholder='Run log path; blank -> <Run Dir>/train.log')
launch_plan_output = widgets.Textarea(
    value='',
    description='Launch Plan',
    disabled=True,
    layout=widgets.Layout(width='100%', height='180px'),
)

sessions_dropdown = widgets.Dropdown(description='Sessions', options=[('<no active sessions>', '')], value='')
tail_lines_input = widgets.BoundedIntText(description='Tail Lines', value=120, min=1, max=5000)
poll_interval_input = widgets.FloatText(description='Poll Secs', value=5.0, layout=widgets.Layout(width='180px'))
auto_refresh_toggle = widgets.ToggleButton(
    value=False,
    description='Auto Refresh',
    icon='refresh',
    layout=widgets.Layout(width='180px'),
)

launch_button = widgets.Button(description='Launch In tmux', button_style='primary')
evaluate_button = widgets.Button(description='Evaluate Run')
refresh_sessions_button = widgets.Button(description='Refresh Sessions')
refresh_log_button = widgets.Button(description='Refresh Log')
end_session_button = widgets.Button(description='End Session', button_style='danger')
clear_output_button = widgets.Button(description='Clear Output')

action_output = widgets.Output(layout=widgets.Layout(overflow_y='auto'))
log_tail_output = widgets.Textarea(
    value='',
    description='',
    disabled=True,
    layout=widgets.Layout(width='100%', height='360px'),
)

_prior_auto_refresh_state = globals().get('_auto_refresh_state')
if isinstance(_prior_auto_refresh_state, dict):
    _prior_stop_event = _prior_auto_refresh_state.get('stop_event')
    _prior_thread = _prior_auto_refresh_state.get('thread')
    if _prior_stop_event is not None:
        _prior_stop_event.set()
    if _prior_thread is not None and _prior_thread.is_alive():
        _prior_thread.join(timeout=1.0)

_model_directory_state = {'last_auto': ''}
_run_id_state = {'last_auto': ''}
_session_name_state = {'last_auto': ''}
_run_dir_state = {'last_auto': ''}
_log_path_state = {'last_auto': ''}
_session_runtime_state = {'last_launched': ''}
_preview_state = {'refreshing': False}
_auto_refresh_state = {'thread': None, 'stop_event': None}


def _append_action(message: str) -> None:
    with action_output:
        print(message)


def _set_banner(target: widgets.HTML, message: str, *, ok: bool, label: str) -> None:
    color = '#0b6f3c' if ok else '#8c1d18'
    target.value = (
        f"<div style='padding:8px 10px;border-left:4px solid {color};'>"
        f"<b>{html.escape(label)}</b><br><code>{html.escape(message)}</code></div>"
    )


def _clear_status() -> None:
    status_html.value = ''


def _sync_auto_text(widget: widgets.Text, suggested: str, state: dict[str, str]) -> None:
    current = str(widget.value or '').strip()
    if (not current) or (current == state['last_auto']):
        widget.value = suggested
    state['last_auto'] = suggested


def _refresh_datasets() -> None:
    discovered = discover_dataset_references(training_root)
    options = [(dataset.name, dataset.name) for dataset in discovered]
    if not options:
        options = [('<no valid datasets discovered>', '')]
    current_train = train_dataset_dropdown.value
    current_validation = validation_dataset_dropdown.value
    train_dataset_dropdown.options = options
    validation_dataset_dropdown.options = options
    valid_values = {value for _, value in options}
    selected_train = current_train if current_train in valid_values else options[0][1]
    selected_validation = current_validation if current_validation in valid_values else selected_train
    train_dataset_dropdown.value = selected_train
    validation_dataset_dropdown.value = selected_validation


def _sync_topology_help() -> None:
    topology_id = str(topology_id_dropdown.value or '').strip()
    if not topology_id:
        topology_help.value = '<div>Select a topology.</div>'
        return
    definition = get_topology_definition(topology_id)
    topology_help.value = (
        f"<div><b>{html.escape(str(definition.topology_metadata.get('display_name', topology_id)))}</b></div>"
        f"<div>Status: <code>{html.escape(str(definition.topology_metadata.get('status', 'active')))}</code></div>"
        f"<div>{html.escape(str(definition.topology_metadata.get('note', '')))}</div>"
    )


def _sync_default_model_directory() -> None:
    model_name = str(model_name_text.value or '').strip() or 'roi-fcn-tiny'
    try:
        suggested = suggest_model_run_id(model_name)
    except Exception:
        return
    _sync_auto_text(model_directory_text, suggested, _model_directory_state)


def _sync_default_run_id() -> None:
    model_directory = str(model_directory_text.value or '').strip()
    if not model_directory:
        return
    try:
        suggested = preview_next_run_id(training_root / 'models', model_directory=model_directory)
    except Exception:
        return
    _sync_auto_text(run_id_text, suggested, _run_id_state)


def _sync_default_session_name() -> None:
    model_directory = str(model_directory_text.value or '').strip()
    run_id = str(run_id_text.value or '').strip()
    if not model_directory or not run_id:
        return
    try:
        suggested = default_session_name(model_directory, run_id)
    except Exception:
        return
    _sync_auto_text(session_name_text, suggested, _session_name_state)


def _build_train_config() -> TrainConfig:
    return TrainConfig(
        training_dataset=str(train_dataset_dropdown.value or '').strip(),
        validation_dataset=str(validation_dataset_dropdown.value or '').strip() or None,
        topology_id=str(topology_id_dropdown.value or '').strip(),
        topology_variant=str(topology_variant_dropdown.value or '').strip(),
        model_name=str(model_name_text.value or '').strip() or 'roi-fcn-tiny',
        model_directory=str(model_directory_text.value or '').strip() or None,
        run_id=str(run_id_text.value or '').strip() or None,
        device=str(device_text.value or '').strip() or None,
        batch_size=int(batch_size.value),
        epochs=int(epochs.value),
        learning_rate=float(learning_rate.value),
        weight_decay=float(weight_decay.value),
        gaussian_sigma_px=float(gaussian_sigma.value),
        early_stopping_patience=int(early_stopping_patience.value),
        progress_log_interval_steps=int(progress_log_interval_steps.value),
        roi_width_px=int(roi_width.value),
        roi_height_px=int(roi_height.value),
        evaluation_max_visual_examples=int(evaluation_max_visual_examples.value),
    )


def _refresh_sessions(*_args, selected: str | None = None) -> None:
    current = selected if selected is not None else sessions_dropdown.value
    try:
        sessions = list_sessions()
    except Exception as exc:
        sessions_dropdown.options = [('<tmux unavailable>', '')]
        sessions_dropdown.value = ''
        return

    if not sessions:
        sessions_dropdown.options = [('<no active sessions>', '')]
        sessions_dropdown.value = ''
        return

    options = [(name, name) for name in sessions]
    sessions_dropdown.options = options
    valid_values = {value for _, value in options}
    sessions_dropdown.value = current if current in valid_values else options[0][1]


def _resolve_log_path() -> str:
    log_path = str(log_path_text.value or '').strip()
    if log_path:
        return log_path
    run_dir = str(run_dir_text.value or '').strip()
    if not run_dir:
        return ''
    return str(Path(run_dir).expanduser().resolve() / 'train.log')


def _refresh_log(*_args) -> None:
    log_path = _resolve_log_path()
    if not log_path:
        log_tail_output.value = '[log path blank]'
        return
    try:
        log_tail_output.value = read_log_tail(log_path, max_lines=int(tail_lines_input.value))
    except Exception as exc:
        log_tail_output.value = traceback.format_exc()


def _refresh_runtime_output(*_args) -> None:
    _refresh_sessions()
    _refresh_log()


def _stop_auto_refresh() -> None:
    stop_event = _auto_refresh_state.get('stop_event')
    thread = _auto_refresh_state.get('thread')
    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive():
        thread.join(timeout=1.0)
    _auto_refresh_state['thread'] = None
    _auto_refresh_state['stop_event'] = None


def _auto_refresh_loop(stop_event: threading.Event) -> None:
    while not stop_event.is_set():
        try:
            _refresh_runtime_output()
        except Exception as exc:
            _append_action(f'Auto refresh failed: {exc}')
        interval = max(0.5, float(poll_interval_input.value))
        stop_event.wait(interval)


def _on_auto_refresh_toggle(change) -> None:
    enabled = bool(change['new'])
    if enabled:
        _stop_auto_refresh()
        _refresh_runtime_output()
        stop_event = threading.Event()
        thread = threading.Thread(target=_auto_refresh_loop, args=(stop_event,), daemon=True)
        _auto_refresh_state['thread'] = thread
        _auto_refresh_state['stop_event'] = stop_event
        thread.start()
        _append_action(f'Auto refresh enabled ({max(0.5, float(poll_interval_input.value)):.1f}s).')
    else:
        _stop_auto_refresh()
        _append_action('Auto refresh disabled.')


def _refresh_launch_preview(*_args) -> None:
    if _preview_state['refreshing']:
        return
    _preview_state['refreshing'] = True
    try:
        _sync_default_model_directory()
        _sync_default_run_id()
        _sync_default_session_name()
        plan = plan_tmux_training_launch(
            training_root,
            _build_train_config(),
            python_executable=PYTHON_EXECUTABLE,
            session_name=str(session_name_text.value or '').strip() or None,
        )
        _sync_auto_text(run_dir_text, plan.run_dir, _run_dir_state)
        _sync_auto_text(log_path_text, plan.log_path, _log_path_state)

        session_note = 'session availability not checked'
        preview_ok = True
        try:
            session_note = 'session already active' if session_exists(plan.session_name) else 'session name available'
            preview_ok = session_note == 'session name available'
        except Exception as exc:
            session_note = f'tmux availability not confirmed: {exc}'
            preview_ok = False

        launch_plan_output.value = '\n'.join(
            [
                f'session_name={plan.session_name}',
                f'model_directory={plan.model_directory}',
                f'run_id={plan.run_id}',
                f'run_dir={plan.run_dir}',
                f'log_path={plan.log_path}',
                f'working_directory={plan.working_directory}',
                f'attach_command=tmux attach -t {plan.session_name}',
                f'session_status={session_note}',
                '',
                plan.command,
            ]
        )
        _set_banner(preview_html, f'Launch preview ready for {plan.session_name}. {session_note}.', ok=preview_ok, label='Preview')
    except Exception as exc:
        launch_plan_output.value = f'[preview unavailable]\n{exc}'
        _set_banner(preview_html, str(exc), ok=False, label='Preview')
    finally:
        _preview_state['refreshing'] = False


def _on_refresh_datasets(_button) -> None:
    _refresh_datasets()
    _refresh_launch_preview()
    _append_action('Refreshed dataset references.')


def _on_topology_id_changed(_change) -> None:
    topology_id = str(topology_id_dropdown.value or '').strip()
    variants = list_topology_variants(topology_id)
    topology_variant_dropdown.options = [(value, value) for value in variants]
    topology_variant_dropdown.value = variants[0]
    _sync_topology_help()
    _refresh_launch_preview()


def _on_launch_clicked(_button) -> None:
    _clear_status()
    try:
        plan = plan_tmux_training_launch(
            training_root,
            _build_train_config(),
            python_executable=PYTHON_EXECUTABLE,
            session_name=str(session_name_text.value or '').strip() or None,
        )
        launch_session(
            plan.session_name,
            plan.command,
            plan.log_path,
            working_directory=plan.working_directory,
        )
        _session_runtime_state['last_launched'] = plan.session_name
        _preview_state['refreshing'] = True
        try:
            _model_directory_state['last_auto'] = plan.model_directory
            _run_id_state['last_auto'] = plan.run_id
            _session_name_state['last_auto'] = plan.session_name
            _run_dir_state['last_auto'] = plan.run_dir
            _log_path_state['last_auto'] = plan.log_path
            model_directory_text.value = plan.model_directory
            run_id_text.value = plan.run_id
            session_name_text.value = plan.session_name
            run_dir_text.value = plan.run_dir
            log_path_text.value = plan.log_path
        finally:
            _preview_state['refreshing'] = False
        _refresh_sessions(selected=plan.session_name)
        _refresh_log()
        _append_action(f'Launched tmux session: {plan.session_name}')
        _append_action(f'Run dir: {plan.run_dir}')
        _append_action(f'Log path: {plan.log_path}')
        _append_action(f'Attach with: tmux attach -t {plan.session_name}')
        _set_banner(status_html, f'Training launched in tmux session {plan.session_name}', ok=True, label='Success')
    except Exception as exc:
        _append_action(traceback.format_exc())
        _refresh_launch_preview()


def _on_evaluate_clicked(_button) -> None:
    _clear_status()
    try:
        run_dir_text_value = str(run_dir_text.value or '').strip()
        if not run_dir_text_value:
            raise ValueError('Run Dir cannot be blank for evaluation.')
        summary = evaluate_saved_run(
            EvalConfig(
                model_run_directory=run_dir_text_value,
                training_dataset=str(train_dataset_dropdown.value or '').strip() or None,
                validation_dataset=str(validation_dataset_dropdown.value or '').strip() or None,
                batch_size=int(batch_size.value),
                roi_width_px=int(roi_width.value),
                roi_height_px=int(roi_height.value),
                device=str(device_text.value or '').strip() or None,
                evaluation_max_visual_examples=int(evaluation_max_visual_examples.value),
            )
        )
        _append_action(str(summary))
        _set_banner(status_html, f'Evaluation completed using checkpoint {summary.get("checkpoint_path")}', ok=True, label='Success')
    except Exception as exc:
        _append_action(traceback.format_exc())


def _on_end_session_clicked(_button) -> None:
    _clear_status()
    candidates: list[str] = []
    for raw in [sessions_dropdown.value, _session_runtime_state.get('last_launched'), session_name_text.value]:
        text = str(raw or '').strip()
        if text and text not in candidates:
            candidates.append(text)
    try:
        if not candidates:
            raise ValueError('Select an active tmux session or enter a session name.')
        for session_name in candidates:
            if end_session(session_name):
                if _session_runtime_state.get('last_launched') == session_name:
                    _session_runtime_state['last_launched'] = ''
                _append_action(f'Ended tmux session: {session_name}')
                _set_banner(status_html, f'Ended tmux session {session_name}', ok=True, label='Success')
                _refresh_sessions()
                _refresh_launch_preview()
                return
        raise ValueError(f'Session not found. Tried: {", ".join(candidates)}')
    except Exception as exc:
        _append_action(traceback.format_exc())


def _on_clear_output_clicked(_button) -> None:
    action_output.clear_output()
    log_tail_output.value = ''
    preview_html.value = ''
    status_html.value = ''


refresh_datasets_button.on_click(_on_refresh_datasets)
refresh_sessions_button.on_click(_refresh_sessions)
refresh_log_button.on_click(_refresh_runtime_output)
auto_refresh_toggle.observe(_on_auto_refresh_toggle, names='value')
launch_button.on_click(_on_launch_clicked)
evaluate_button.on_click(_on_evaluate_clicked)
end_session_button.on_click(_on_end_session_clicked)
clear_output_button.on_click(_on_clear_output_clicked)
topology_id_dropdown.observe(_on_topology_id_changed, names='value')

for widget in [
    train_dataset_dropdown,
    validation_dataset_dropdown,
    topology_variant_dropdown,
    model_name_text,
    model_directory_text,
    run_id_text,
    session_name_text,
    device_text,
    batch_size,
    epochs,
    learning_rate,
    weight_decay,
    gaussian_sigma,
    early_stopping_patience,
    progress_log_interval_steps,
    roi_width,
    roi_height,
    evaluation_max_visual_examples,
]:
    widget.observe(_refresh_launch_preview, names='value')

_refresh_datasets()
_sync_topology_help()
_refresh_sessions()
_refresh_launch_preview()

panel = widgets.VBox(
    [
        widgets.HTML(f"<b>ROI-FCN Training Control Panel</b> <code>{TMUX_CONTROL_PANEL_BUILD_V02}</code>"),
        widgets.HBox([train_dataset_dropdown, validation_dataset_dropdown, refresh_datasets_button]),
        widgets.HBox([topology_id_dropdown, topology_variant_dropdown]),
        topology_help,
        widgets.HBox([model_name_text, model_directory_text, run_id_text]),
        widgets.HBox([session_name_text]),
        widgets.HBox([device_text, batch_size, epochs, learning_rate]),
        widgets.HBox([weight_decay, gaussian_sigma, early_stopping_patience, progress_log_interval_steps]),
        widgets.HBox([roi_width, roi_height, evaluation_max_visual_examples]),
        run_dir_text,
        log_path_text,
        launch_plan_output,
        preview_html,
        widgets.HBox([launch_button, evaluate_button, refresh_sessions_button, refresh_log_button, end_session_button, clear_output_button]),
        widgets.HBox([sessions_dropdown, tail_lines_input, poll_interval_input, auto_refresh_toggle]),
        status_html,
        action_output,
        log_tail_output,
    ]
)

display(panel)
